<a href="https://colab.research.google.com/github/AngelaOrtiz25/tesis_angela/blob/gh-pages/datos_codigos/recolecion_de_datos_pagerank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalacion de la paqueteria

In [ ]:
# Instalar paquetes primero
!pip install rasterio geopandas pyproj folium

## Genera el mapa de los municipios de Chiapas con las rutas que se calcula las distancias con la altitud o elevación

In [ ]:
from google.colab import drive
import requests
import csv
import time
from urllib.parse import quote
import folium
import math
import rasterio
from rasterio.merge import merge
from shapely.geometry import LineString
import numpy as np
from pyproj import Geod
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Montar Google Drive
drive.mount('/content/drive')

# Instalar librerías necesarias
!pip install rasterio geopandas pyproj folium

# Configuración de ciudades
ciudades = [
"Acacoyagua", "Acala", "Acapetahua", "Aldama", "Altamirano", "Amatán", "Amatenango de la Frontera", "Amatenango del Valle", "Ángel Albino Corzo",
"Arriaga", "Bejucal de Ocampo", "Bella Vista", "Benemérito de las Américas", "Berriozábal", "Bochil", "Cacahoatán", "Capitán Luis Ángel Vidal",
"Catazajá", "Chalchihuitán", "Chamula", "Chanal", "Chapultenango","Chenalhó", "Chiapa de Corzo", "Chiapilla", "Chicoasén", "Chicomuselo", "Chilón",
"Cintalapa de Figueroa", "Coapilla", "Comitán de Domínguez", "Copainalá", "El Bosque", "El Parral","El Porvenir", "Emiliano Zapata", "Escuintla",
"Francisco León", "Frontera Comalapa", "Frontera Hidalgo", "Honduras de la Sierra", "Huehuetán", "Huitiupán", "Huixtán", "Huixtla", "Ixhuatán",
"Ixtacomitán", "Ixtapa", "Ixtapangajoya", "Jiquipilas", "Jitotol", "Juárez", "La Concordia", "La Grandeza", "La Independencia", "La Libertad",
"La Trinitaria", "Larráinzar", "Las Margaritas", "Las Rosas", "Mapastepec", "Maravilla Tenejapa", "Marqués de Comillas", "Mazapa de Madero", "Mazatán",
"Metapa", "Mezcalapa", "Mitontic", "Montecristo de Guerrero", "Motozintla", "Nicolás Ruíz", "Ocosingo", "Ocotepec", "Ocozocoautla de Espinosa",
"Ostuacán", "Osumacinta", "Oxchuc", "Palenque", "Pantelhó", "Pantepec", "Pichucalco", "Pijijiapan", "Pueblo Nuevo Solistahuacán", "Rayón", "Reforma",
"Rincón Chamula San Pedro", "Sabanilla", "Salto de Agua", "San Andrés Duraznal", "San Cristóbal de las Casas", "San Fernando", "San Juan Cancuc",
"San Lucas", "Santiago el Pinar", "Siltepec", "Simojovel", "Sitalá", "Socoltenango", "Solosuchiapa", "Soyaló", "Suchiapa", "Suchiate", "Sunuapa",
"Tapachula", "Tapalapa", "Tapilula", "Tecpatán", "Tenejapa", "Teopisca", "Tila", "Tonalá", "Totolapa", "Tumbalá",  "Tuxtla Chico", "Tuxtla Gutiérrez",
"Tuzantán", "Tzimol", "Unión Juárez", "Venustiano Carranza",  "Villa Comaltitlán", "Villa Corzo", "Villaflores", "Yajalón", "Zinacantán"
]

ciudades.sort()

# Funciones auxiliares
def cargar_mosaico_srtm():
    archivos_hgt = [
        "/content/drive/MyDrive/tesis/chiapas_hgt/N17W094.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N17W093.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N17W092.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N16W095.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N16W094.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N16W093.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N15W094.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N15W092.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N14W093.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N16W091.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N15W093.hgt",
        "/content/drive/MyDrive/tesis/chiapas_hgt/N16W092.hgt"
    ]
    src_files = [rasterio.open(f) for f in archivos_hgt]
    mosaico, transform = merge(src_files)
    for src in src_files:
        src.close()
    return mosaico, transform

def obtener_coordenadas(ciudad, intentos=3):
    """Obtiene coordenadas con reintentos y manejo de errores"""
    for intento in range(intentos):
        try:
            url = f"https://nominatim.openstreetmap.org/search?q={quote(ciudad+', Chiapas, México')}&format=json"
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}, timeout=10)

            # Verificar si la respuesta es válida
            if response.status_code == 200:
                try:
                    data = response.json()
                    if data and isinstance(data, list) and len(data) > 0:
                        return (float(data[0]['lat']), float(data[0]['lon']))
                    else:
                        print(f" {ciudad}: No se encontraron datos")
                        return None
                except requests.exceptions.JSONDecodeError:
                    print(f" {ciudad}: Error JSON - Respuesta no es JSON válido (intento {intento+1}/{intentos})")
            elif response.status_code == 429:
                print(f" {ciudad}: Rate limiting - Esperando 5 segundos...")
                time.sleep(5)
            else:
                print(f" {ciudad}: Error HTTP {response.status_code} (intento {intento+1}/{intentos})")

            time.sleep(2)  # Esperar antes de reintentar

        except requests.exceptions.RequestException as e:
            print(f" {ciudad}: Error de conexión - {str(e)[:50]} (intento {intento+1}/{intentos})")
            time.sleep(3)

    print(f"  ✗ {ciudad}: No se pudieron obtener coordenadas después de {intentos} intentos")
    return None

def obtener_ruta_osrm(coord_origen, coord_destino):
    lon1, lat1 = coord_origen[1], coord_origen[0]
    lon2, lat2 = coord_destino[1], coord_destino[0]
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                distancia = data['routes'][0]['distance']
                duracion = data['routes'][0]['duration']
                geometria = data['routes'][0]['geometry']['coordinates']
                return distancia, duracion, [(lat, lon) for lon, lat in geometria]
        else:
            print(f"  Error OSRM: HTTP {response.status_code}")
    except Exception as e:
        print(f"  Error OSRM: {e}")
    return None, None, None

def calcular_distancia_3d_ruta(mosaico, transform, ruta_coords):
    geod = Geod(ellps="WGS84")
    distancia_2d = 0
    distancia_3d = 0
    elevaciones = []

    # Obtener elevaciones
    for coord in ruta_coords:
        lon, lat = coord[1], coord[0]
        try:
            col, row = ~transform * (lon, lat)
            row_int, col_int = int(round(row)), int(round(col))
            if 0 <= row_int < mosaico.shape[1] and 0 <= col_int < mosaico.shape[2]:
                elev = float(mosaico[0, row_int, col_int])
                elevaciones.append(elev)
            else:
                elevaciones.append(np.nan)
        except:
            elevaciones.append(np.nan)

    # Calcular distancias 3D
    for i in range(1, len(ruta_coords)):
        lon1, lat1 = ruta_coords[i-1][1], ruta_coords[i-1][0]
        lon2, lat2 = ruta_coords[i][1], ruta_coords[i][0]
        elev1, elev2 = elevaciones[i-1], elevaciones[i]

        if np.isnan(elev1) or np.isnan(elev2):
            continue

        _, _, dist2d = geod.inv(lon1, lat1, lon2, lat2)
        delta_elev = elev2 - elev1
        dist3d = math.sqrt(dist2d**2 + delta_elev**2)
        distancia_2d += dist2d
        distancia_3d += dist3d

    return distancia_2d, distancia_3d, elevaciones

def graficar_perfil(ruta_coords, elevaciones):
    geod = Geod(ellps="WGS84")
    distancias = [0]
    for i in range(1, len(ruta_coords)):
        lon1, lat1 = ruta_coords[i-1][1], ruta_coords[i-1][0]
        lon2, lat2 = ruta_coords[i][1], ruta_coords[i][0]
        _, _, dist = geod.inv(lon1, lat1, lon2, lat2)
        distancias.append(distancias[-1] + dist/1000)
    plt.figure(figsize=(12, 5))
    plt.plot(distancias, elevaciones, 'b-', linewidth=1)
    plt.title('Perfil de Elevación de la Ruta')
    plt.xlabel('Distancia (km)')
    plt.ylabel('Elevación (m)')
    plt.grid(True)
    plt.show()

# Proceso principal
print("Cargando mosaico SRTM...")
mosaico, transform = cargar_mosaico_srtm()

print("\nObteniendo coordenadas de municipios...")
print("="*50)
coordenadas = {}
for i, ciudad in enumerate(ciudades):
    print(f"[{i+1}/{len(ciudades)}] {ciudad}...", end=" ")
    coord = obtener_coordenadas(ciudad)
    if coord:
        coordenadas[ciudad] = coord
        print(f"✓ {coord[0]:.4f}, {coord[1]:.4f}")
    else:
        print("✗ No encontrada")
    time.sleep(1.5)  # Respetar límite de la API

print(f"\n✓ Coordenadas obtenidas: {len(coordenadas)}/{len(ciudades)} municipios")

archivo_salida = "/content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completo.csv"


# Definir la ruta donde guardarás el mapa
ruta_drive = "/content/drive/MyDrive/tesis"


# Crear mapa base
mapa = folium.Map(location=[16.5, -92.5], zoom_start=7)
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen']

with open(archivo_salida, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Origen", "Destino", "Distancia carretera (km)", "Distancia 3D (km)", "Diferencia (km)", "Tiempo estimado (min)"])

    total_pares = 0
    pares_procesados = 0

    for i in range(len(ciudades)):
        for j in range(len(ciudades)):
            if i == j:
                continue
            total_pares += 1

    for i in range(len(ciudades)):
        for j in range(len(ciudades)):
            if i == j:
                continue

            origen = ciudades[i]
            destino = ciudades[j]

            if origen in coordenadas and destino in coordenadas:
                pares_procesados += 1
                print(f"\n[{pares_procesados}/{total_pares}] {origen} → {destino}...")

                coord_origen = coordenadas[origen]
                coord_destino = coordenadas[destino]

                distancia_ruta, duracion, ruta_coords = obtener_ruta_osrm(coord_origen, coord_destino)

                if ruta_coords and distancia_ruta:
                    dist2d, dist3d, elevaciones = calcular_distancia_3d_ruta(mosaico, transform, ruta_coords)

                    if dist2d > 0 and dist3d > 0:
                        dist_ruta_km = distancia_ruta/1000
                        dist3d_km = dist3d/1000
                        diferencia = dist3d_km - dist_ruta_km
                        tiempo_min = duracion/60

                        print(f"   Distancia carretera: {dist_ruta_km:.2f} km")
                        print(f"   Distancia 3D: {dist3d_km:.2f} km (+{diferencia:.2f} km)")
                        print(f"   Tiempo estimado: {tiempo_min:.1f} min")

                        writer.writerow([origen, destino, round(dist_ruta_km, 3), round(dist3d_km, 3), round(diferencia, 3), round(tiempo_min, 1)])

                        # Agregar ruta al mapa
                        color = colores[(i+j) % len(colores)]
                        folium.PolyLine(locations=ruta_coords, color=color, weight=2, opacity=0.6,
                                        tooltip=f"{origen} → {destino}<br>Distancia: {dist_ruta_km:.1f}km<br>Tiempo: {tiempo_min:.1f}min").add_to(mapa)
                else:
                    print(f"   No se pudo calcular la ruta")

                time.sleep(0.5)  # Pequeña pausa para no saturar OSRM

        # Agregar marcadores de ciudades después de procesar rutas
        for ciudad, coord in coordenadas.items():
            folium.Marker(location=coord, popup=ciudad, icon=folium.Icon(color='blue', icon='info-sign')).add_to(mapa)

print(f"\n Datos guardados en {archivo_salida}")
print(f" Mapa generado con {pares_procesados} rutas")

# 4. GUARDAR EL MAPA COMO HTML EN GOOGLE DRIVE
nombre_archivo_html = 'mapa_rutas_chiapas_completo.html'
ruta_completa_html = os.path.join(ruta_drive, nombre_archivo_html)

mapa.save(ruta_completa_html)
print(f" Mapa guardado en: {ruta_completa_html}")

# Mostrar mapa en Colab
display(mapa)

# Mostrar CSV resultante
df = pd.read_csv(archivo_salida)
print(f"\n Total de rutas calculadas: {len(df)}")
display(df.head(10))

# Resumen estadístico
print("\n Resumen estadístico:")
print(f"  Distancia carretera promedio: {df['Distancia carretera (km)'].mean():.2f} km")
print(f"  Distancia 3D promedio: {df['Distancia 3D (km)'].mean():.2f} km")
print(f"  Diferencia promedio: {df['Diferencia (km)'].mean():.2f} km")
print(f"  Tiempo promedio: {df['Tiempo estimado (min)'].mean():.1f} min")

Cambia las letras con tilde sin tilde y la ñ con la n

In [ ]:
import pandas as pd
import unicodedata

# Montar Google Drive si no lo has hecho
from google.colab import drive
drive.mount('/content/drive')

# Ruta del archivo original en Google Drive
archivo_original = '/content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completo.csv'
archivo_limpio = '/content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completolimpio.csv'

# Función para quitar tildes y reemplazar ñ
def limpiar_texto(texto):
    if isinstance(texto, str):
        texto = unicodedata.normalize('NFD', texto)
        texto = texto.encode('ascii', 'ignore').decode('utf-8')
        texto = texto.replace('ñ', 'n').replace('Ñ', 'N')
    return texto

# Leer el CSV
df = pd.read_csv(archivo_original)

# Limpiar los nombres de columnas también (opcional pero recomendable)
df.columns = [limpiar_texto(col) for col in df.columns]

# Aplicar la limpieza a todas las celdas de texto
df_limpio = df.applymap(limpiar_texto)

# Guardar el archivo limpio
df_limpio.to_csv(archivo_limpio, index=False)

print("Archivo limpio guardado como:", archivo_limpio)

Mounted at /content/drive


/tmp/ipykernel_646/2138118491.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_limpio = df.applymap(limpiar_texto)


Archivo limpio guardado como: /content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completolimpio.csv


## Obtiene la matriz de red

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta al archivo CSV
ruta = '/content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completolimpio.csv'

# Cargar CSV
df = pd.read_csv(ruta, index_col=0)
print(" Archivo cargado:")
print(df.head())

# --- Extraer solo columnas necesarias para la matriz ---

# Convertir a matriz tipo [origen x destino] con distancia 3D como valor
df_matriz = df.pivot_table(index='Origen', columns='Destino', values='Distancia 3D (km)')
print(" Matriz de distancias 3D creada:")
print(df_matriz.head())

# Rellenar NaN con 0 (no hay conexión directa)
df_matriz = df_matriz.fillna(0)

# Reemplazar ceros en la diagonal si existen (para evitar división por cero)
np.fill_diagonal(df_matriz.values, np.nan)

# --- Construir la matriz de transición ---

# Invertir distancias (menor distancia = mayor probabilidad)
matriz_inversa = 1 / df_matriz

# Limpiar infinitos y NaNs
matriz_inversa = matriz_inversa.replace([np.inf, -np.inf], np.nan).fillna(0)

# Normalizar por fila
matriz_transicion = matriz_inversa.div(matriz_inversa.sum(axis=1), axis=0)

# Asegurar que índice y columnas coincidan
municipios = matriz_transicion.index.tolist()
matriz_transicion = matriz_transicion.reindex(index=municipios, columns=municipios, fill_value=0)

# Verificar forma final
print("Matriz de transición final:")
print(matriz_transicion.round(3))

# Guardar CSV
salida = '/content/drive/MyDrive/tesis/matriz_transicion_3d.csv'
matriz_transicion.to_csv(salida)
print(f"Matriz de transición guardada en: {salida}")

# Leer y mostrar el CSV guardado para verificación en Colab
df_guardado = pd.read_csv(salida, index_col=0)
print("Archivo guardado cargado para mostrar:")
display(df_guardado)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Archivo cargado:
                              Destino  Distancia carretera (km)  \
Origen                                                            
Acacoyagua                      Acala                   347.113   
Acacoyagua                 Acapetahua                    33.390   
Acacoyagua                     Aldama                   388.588   
Acacoyagua                 Altamirano                   350.590   
Acacoyagua  Amatenango de la Frontera                   146.018   

            Distancia 3D (km)  Diferencia (km)  Tiempo estimado (min)  
Origen                                                                 
Acacoyagua            347.931            0.818                  284.9  
Acacoyagua             33.572            0.182                   58.0  
Acacoyagua            390.375            1.787                  334.7  
Acacoyagua            3

,Acacoyagua,Acala,Acapetahua,Aldama,Altamirano,Amatan,Amatenango de la Frontera,Amatenango del Valle,Angel Albino Corzo,Arriaga,...,Tuxtla Gutierrez,Tuzantan,Tzimol,Union Juarez,Venustiano Carranza,Villa Comaltitlan,Villa Corzo,Villaflores,Yajalon,Zinacantan
Origen,,,,,,,,,,,,,,,,,,,,,
Acacoyagua,0.000000,0.004928,0.051071,0.004392,0.004835,0.003543,0.011454,0.005627,0.006600,0.009955,...,0.005782,0.029485,0.006471,0.012439,0.004531,0.039601,0.005607,0.006347,0.003386,0.004979
Acala,0.002685,0.000000,0.002616,0.010155,0.005268,0.004535,0.004416,0.012792,0.005345,0.005262,...,0.017392,0.002449,0.008168,0.002969,0.021004,0.002547,0.007114,0.009161,0.004517,0.020103
Acapetahua,0.047185,0.004601,0.000000,0.004120,0.004795,0.003348,0.011628,0.005596,0.006585,0.008908,...,0.005359,0.032002,0.006454,0.012673,0.005569,0.065959,0.005205,0.005854,0.003204,0.004646
Aldama,0.002231,0.009498,0.002179,0.000000,0.005570,0.005940,0.003602,0.011302,0.003492,0.003966,...,0.009130,0.002677,0.006354,0.002523,0.007275,0.002128,0.004414,0.005187,0.005970,0.015095
Altamirano,0.004553,0.008987,0.004589,0.010237,0.000000,0.006978,0.007455,0.015574,0.004762,0.005227,...,0.008804,0.005363,0.014606,0.005030,0.008727,0.004925,0.005638,0.006291,0.013089,0.010936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Villa Comaltitlan,0.033518,0.004031,0.059227,0.003619,0.004609,0.002954,0.012411,0.005443,0.006500,0.007624,...,0.004675,0.050882,0.006358,0.013757,0.005415,0.000000,0.004545,0.005093,0.003359,0.004069
Villa Corzo,0.005383,0.012569,0.005225,0.008375,0.005844,0.005699,0.007960,0.008116,0.013514,0.012103,...,0.014442,0.004852,0.007135,0.003939,0.010223,0.005071,0.000000,0.035113,0.005292,0.010889
Villaflores,0.005224,0.013907,0.005052,0.008453,0.005602,0.005448,0.006319,0.008148,0.010191,0.014025,...,0.016695,0.004649,0.007020,0.003694,0.010734,0.004885,0.030188,0.000000,0.005018,0.011601


## PageRank

In [ ]:
!pip install networkx==2.8.8

In [ ]:
import networkx as nx

# Crear grafo dirigido desde la matriz de transición usando nombres reales
G = nx.from_pandas_adjacency(matriz_transicion, create_using=nx.DiGraph)

# Calcular PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85)

# Mostrar resultados ordenados por puntaje (de mayor a menor)
pagerank_ordenado = dict(sorted(pagerank_scores.items(), key=lambda item: item[1], reverse=True))
print("Ranking de ciudades según PageRank:")
for ciudad, score in pagerank_ordenado.items():
    print(f"{ciudad}: {score:.4f}")

# Guardar top 10 en CSV (crea el archivo si no existe, lo sobrescribe si ya existe)
top_10 = list(pagerank_ordenado.items())[:15]
df_top10 = pd.DataFrame(top_10, columns=['Municipio', 'Score_PageRank'])
df_top10.to_csv('/content/drive/MyDrive/tesis/top10_pagerank.csv', index=False, encoding='utf-8')
print("\n✓ Top 10 guardado en 'top10_pagerank.csv'")

📊 Ranking de ciudades según PageRank:
San Cristobal de las Casas: 0.0112
Rayon: 0.0112
Chamula: 0.0111
Chiapilla: 0.0110
Pueblo Nuevo Solistahuacan: 0.0109
Aldama: 0.0109
Rincon Chamula San Pedro: 0.0108
Pantepec: 0.0107
Santiago el Pinar: 0.0106
Larrainzar: 0.0105
Teopisca: 0.0104
Totolapa: 0.0104
Tapilula: 0.0103
Acala: 0.0102
Mitontic: 0.0102
Bochil: 0.0100
Tuxtla Gutierrez: 0.0100
Chenalho: 0.0100
Chiapa de Corzo: 0.0099
Ixtapa: 0.0099
San Lucas: 0.0099
Zinacantan: 0.0098
Jitotol: 0.0098
Tenejapa: 0.0097
Amatenango del Valle: 0.0096
Soyalo: 0.0095
Osumacinta: 0.0095
Ixhuatan: 0.0095
Huixtan: 0.0095
Copainala: 0.0094
El Bosque: 0.0094
San Andres Duraznal: 0.0094
Emiliano Zapata: 0.0093
Chalchihuitan: 0.0093
Tapalapa: 0.0092
Pantelho: 0.0092
Comitan de Dominguez: 0.0091
Nicolas Ruiz: 0.0091
Coapilla: 0.0091
San Fernando: 0.0091
Chicoasen: 0.0091
Las Rosas: 0.0090
Suchiapa: 0.0088
San Juan Cancuc: 0.0088
Ixtacomitan: 0.0088
Venustiano Carranza: 0.0088
Oxchuc: 0.0086
Pichucalco: 0.0086